# Demos

This is the notebook containing the demos for Feature Store, Model Monitor, and Clarify. Testing for these exercises was performed using __2 vCPU + 4 GiB notebook instance with Python 3 (TensorFlow 2.1 Python 3.6 CPU Optimized) kernel__.

## Staging

We'll begin by initializing some variables that are used throughout the demos. These are often assumed to be present in code samples you'll find in the AWS documenation.

In [1]:
# import sagemaker
# from sagemaker.session import Session
# from sagemaker import get_execution_role

# role = get_execution_role()
# session = sagemaker.Session()
# region = session.boto_region_name
# bucket = session.default_bucket()

In [2]:
from sagemaker.core.helper.session_helper import Session, get_execution_role

role = get_execution_role()
session = Session()
region = session.boto_region_name
bucket = session.default_bucket()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Feature Store
---

Feature store is a special database to give ML systems a consistent data flow across training and inference workloads. It can ingest data in batches (for training) as well as serve input features to models with very low latency for real-time prediction.

For this demo we'll use the boston housing dataset, which you can learn more about here: https://www.cs.toronto.edu/~delve/data/boston/bostonDetail.html

In [4]:
from tensorflow.keras.datasets import boston_housing

(x_train, y_train), (x_test, y_test) = boston_housing.load_data(test_split=0.1, seed=1234)

# Manually add headers
train_headers = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT"]
test_headers = ["MEDV"]

2026-10-06 08:33:36.165442: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
import pandas as pd
import time
import uuid


boston_train = pd.DataFrame(x_train, columns=train_headers)

Once we have our data, we can create a feature group. Remember to attach event time and ID columns - Feature Store needs them.

In [7]:
# boston_train["EventTime"] = time.time()
# boston_train["id"] = range(len(boston_train))

# # Create feature group
# from sagemaker.feature_store.feature_group import FeatureGroup

# feature_group = FeatureGroup(
#     name="boston-features", sagemaker_session=session
# )

# # Load Feature definitions
# feature_group.load_feature_definitions(data_frame=boston_train)

In [8]:
import time
from sagemaker.core.resources import FeatureGroup
from sagemaker.core.shapes import FeatureDefinition

boston_train["EventTime"] = time.time()
boston_train["id"] = range(len(boston_train))

In [12]:
# Checking the datatype to prepare the datatype feature definition function (similar to load_feature_definitions() which doesn't exist anymore)
boston_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 455 entries, 0 to 454
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CRIM       455 non-null    float64
 1   ZN         455 non-null    float64
 2   INDUS      455 non-null    float64
 3   CHAS       455 non-null    float64
 4   NOX        455 non-null    float64
 5   RM         455 non-null    float64
 6   AGE        455 non-null    float64
 7   DIS        455 non-null    float64
 8   RAD        455 non-null    float64
 9   TAX        455 non-null    float64
 10  PTRATIO    455 non-null    float64
 11  B          455 non-null    float64
 12  LSTAT      455 non-null    float64
 13  EventTime  455 non-null    float64
 14  id         455 non-null    int64  
dtypes: float64(14), int64(1)
memory usage: 53.4 KB


In [13]:
# Build the definitions from the DataFrame yourself

def to_feature_type(dtype):
    name = str(dtype)
    if name.startswith("int"):
        return "Integral"
    if name.startswith("float"):
        return "Fractional"
    if name == "object":
        return "String"
    raise ValueError(f"Unsupported dtype: {name}")

In [14]:
feature_definitions = [
    FeatureDefinition(
        feature_name=col,
        feature_type=to_feature_type(boston_train[col].dtype),
    )
    for col in boston_train.columns
]
feature_definitions

[FeatureDefinition(feature_name='CRIM', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='ZN', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='INDUS', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='CHAS', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='NOX', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='RM', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='AGE', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='DIS', feature_type='Fractional', collection_type=Unassigned(),

The feature group is not created until we call the `create` method, let's do that now:

In [24]:
# feature_group.create(
#     s3_uri=f"s3://{bucket}/features",
#     record_identifier_name='id',
#     event_time_feature_name="EventTime",
#     role_arn=role,
#     enable_online_store=True,
# ) <- deprecated

In [26]:
# # debug 
# from sagemaker.core.shapes import (
#     OnlineStoreConfig, OfflineStoreConfig, S3StorageConfig,
# )

# FeatureGroup.get(feature_group_name="boston-features").delete()

In [27]:
from sagemaker.core.shapes import OnlineStoreConfig

# Create feature group 
feature_group = FeatureGroup.create(
    feature_group_name="boston-features",
    record_identifier_feature_name="id",
    event_time_feature_name="EventTime",
    feature_definitions=feature_definitions,
    role_arn=role,
    online_store_config=OnlineStoreConfig(enable_online_store=True),  # To serve inferences in real time
    offline_store_config=OfflineStoreConfig(
        s3_storage_config=S3StorageConfig(
            s3_uri=f"s3://{bucket}/lesson4/features",
        ),
    ),
)

[10/03/26 10:16:27] INFO     Creating feature_group resource.                                    ]8;id=2361381;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2361382;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11769\11769]8;;\

For applications, we can create a lightweight client to retrieve data with low latency:

In [28]:
runtime = session.boto_session.client(
  'sagemaker-featurestore-runtime',
  region_name=region
)

data = runtime.get_record(
    FeatureGroupName="boston-features",
    RecordIdentifierValueAsString="0"
)

If we try to get records before we ingest any data, the response comes back empty:

In [30]:
# Since we didn't ingest any data yet, we get an empty json object. As you can see below, there is no "Record" key in the response
data

{'ResponseMetadata': {'RequestId': '2fb249d3-306f-468c-bc48-de5871726c72',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '2fb249d3-306f-468c-bc48-de5871726c72',
   'content-type': 'application/json',
   'content-length': '32',
   'date': 'Sat, 03 Oct 2026 10:21:19 GMT'},
  'RetryAttempts': 0}}

In [32]:
# feature_group.ingest(data_frame=boston_train, max_workers=3, wait=True) <- deprecated

In [34]:
from concurrent.futures import ThreadPoolExecutor

# Ingesting data using 3 workers similarly to the deprecated example
records = boston_train.to_dict(orient="records")

def put_row(rec):
    runtime.put_record(
        FeatureGroupName="boston-features",
        Record=[
            {"FeatureName": col, "ValueAsString": str(val)}
            for col, val in rec.items()
        ],
    )

with ThreadPoolExecutor(max_workers=3) as ex:
    list(ex.map(put_row, records))

In [35]:
data = runtime.get_record(
    FeatureGroupName="boston-features",
    RecordIdentifierValueAsString="0"
)
data

{'ResponseMetadata': {'RequestId': 'e82fd6f4-84ad-4f65-b25f-7f32e9cb97ee',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e82fd6f4-84ad-4f65-b25f-7f32e9cb97ee',
   'content-type': 'application/json',
   'content-length': '1110',
   'date': 'Sat, 03 Oct 2026 10:31:43 GMT'},
  'RetryAttempts': 0},
 'Record': [{'FeatureName': 'CRIM', 'ValueAsString': '0.01951'},
  {'FeatureName': 'ZN', 'ValueAsString': '17.5'},
  {'FeatureName': 'INDUS', 'ValueAsString': '1.38'},
  {'FeatureName': 'CHAS', 'ValueAsString': '0.0'},
  {'FeatureName': 'NOX', 'ValueAsString': '0.4161'},
  {'FeatureName': 'RM', 'ValueAsString': '7.104'},
  {'FeatureName': 'AGE', 'ValueAsString': '59.5'},
  {'FeatureName': 'DIS', 'ValueAsString': '9.2229'},
  {'FeatureName': 'RAD', 'ValueAsString': '3.0'},
  {'FeatureName': 'TAX', 'ValueAsString': '216.0'},
  {'FeatureName': 'PTRATIO', 'ValueAsString': '18.6'},
  {'FeatureName': 'B', 'ValueAsString': '393.24'},
  {'FeatureName': 'LSTAT', 'ValueAsString': '8.05'}

## Model Monitor

In this demo we create a monitoring schedule for a deployed model. We'll begin by reloading our data from the previous demo.

In [6]:
from tensorflow.keras.datasets import boston_housing
import pandas as pd

(x_train, y_train), (x_test, y_test) = boston_housing.load_data(test_split=0.1, seed=1234)
headers = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT"]


train = pd.DataFrame(x_train, columns=headers)
train["MEDV"] = y_train

# Target variable must come first per https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html
train.set_index(train.pop('MEDV'), inplace=True)
train.reset_index(inplace=True)
train

,MEDV,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
0,33.0,0.01951,17.5,1.38,0.0,0.4161,7.104,59.5,9.2229,3.0,216.0,18.6,393.24,8.05
1,27.5,0.14866,0.0,8.56,0.0,0.5200,6.727,79.9,2.7778,5.0,384.0,20.9,394.76,9.42
2,5.6,25.04610,0.0,18.10,0.0,0.6930,5.987,100.0,1.5888,24.0,666.0,20.2,396.90,26.77
3,21.2,3.67367,0.0,18.10,0.0,0.5830,6.312,51.9,3.9917,24.0,666.0,20.2,388.62,10.58
4,14.9,9.51363,0.0,18.10,0.0,0.7130,6.728,94.1,2.4961,24.0,666.0,20.2,6.68,18.71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,17.9,18.81100,0.0,18.10,0.0,0.5970,4.628,100.0,1.5539,24.0,666.0,20.2,28.79,34.37
451,14.5,8.49213,0.0,18.10,0.0,0.5840,6.348,86.1,2.0527,24.0,666.0,20.2,83.45,17.64
452,12.7,4.66883,0.0,18.10,0.0,0.7130,5.976,87.9,2.5806,24.0,666.0,20.2,10.48,19.01
453,17.8,0.31827,0.0,9.90,0.0,0.5440,5.914,83.2,3.9986,4.0,304.0,18.4,390.70,18.33


In [3]:
test =  pd.DataFrame(x_test)
test["MEDV"] = y_test

# Target variable must come first per https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html
test.set_index(test.pop('MEDV'), inplace=True)
test.reset_index(inplace=True)

Now we'll upload the data to S3 as train and validation data, then train a model:

In [4]:
# train.to_csv("train.csv", header=False, index=False)
# test.to_csv("validation.csv", header=False, index=False)

# val_location = session.upload_data('./validation.csv', key_prefix="data")
# train_location = session.upload_data('./train.csv', key_prefix="data")

# s3_input_train = sagemaker.inputs.TrainingInput(s3_data=train_location, content_type='csv')
# s3_input_validation = sagemaker.inputs.TrainingInput(s3_data=val_location, content_type='csv')

from sagemaker.core.training.configs import InputData

train.to_csv("train.csv", header=False, index=False)
test.to_csv("validation.csv", header=False, index=False)

val_location = session.upload_data('./validation.csv', key_prefix="data")
train_location = session.upload_data('./train.csv', key_prefix="data")

s3_input_train = InputData(
    channel_name="train", data_source=train_location, content_type="text/csv"
)
s3_input_validation = InputData(
    channel_name="validation", data_source=val_location, content_type="text/csv"
)

In [5]:
# from sagemaker.model_monitor import DataCaptureConfig

# algo_image = sagemaker.image_uris.retrieve("xgboost", region, version='latest')
# s3_output_location = f"s3://{bucket}/models/boston_model"

# model=sagemaker.estimator.Estimator(
#     image_uri=algo_image,
#     role=role,
#     instance_count=1,
#     instance_type='ml.m4.xlarge',
#     volume_size=5,
#     output_path=s3_output_location,
#     sagemaker_session=sagemaker.Session()
# )

# model.set_hyperparameters(max_depth=5,
#                         eta=0.2,
#                         gamma=4,
#                         min_child_weight=6,
#                         subsample=0.8,
#                         objective='reg:linear',
#                         early_stopping_rounds=10,
#                         num_round=200)


# model.fit({'train': s3_input_train, 'validation': s3_input_validation})

from sagemaker.core import image_uris
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import Compute, OutputDataConfig

algo_image = image_uris.retrieve("xgboost", region, version='latest')
s3_output_location = f"s3://{bucket}/models/boston_model"

model = ModelTrainer(
    training_image=algo_image,
    role=role,
    compute=Compute(instance_type="ml.m4.xlarge", instance_count=1, volume_size_in_gb=5),
    output_data_config=OutputDataConfig(s3_output_path=s3_output_location),
    hyperparameters={
        "max_depth": "5", "eta": "0.2", "gamma": "4",
        "min_child_weight": "6", "subsample": "0.8",
        "objective": "reg:linear", "early_stopping_rounds": "10",
        "num_round": "200",
    },
)

model.train(input_data_config=[s3_input_train, s3_input_validation], wait=True)

[10/04/26 10:29:36] INFO     Ignoring unnecessary instance type: None.                            ]8;id=1059144;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=1059145;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=1059152;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=1059153;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Base name not provided. Using default name:                             ]8;id=1059159;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=1059160;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#90\90]8;;\
                             xgboost-job                                                                           

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=1059166;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=1059167;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=1059173;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=1059174;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

...

                    INFO     xgboost-job-20261004102936/algo-1-1791109825:                       ]8;id=1060190;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060191;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [10:32:00] src/tree/updater_prune.cc:74: tree pruning end, 1 roots,                   
                             12 extra nodes, 18 pruned nodes, max_depth=5                                          

                    INFO     xgboost-job-20261004102936/algo-1-1791109825:                       ]8;id=1060196;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060197;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [76]#011train-rmse:0.845425#011validation-rmse:5.5454                                 

                    INFO     xgboost-job-20261004102936/algo-1-1791109825:                       ]8;id=1060202;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060203;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Stopping. Best iteration:                                                             

                    INFO     xgboost-job-20261004102936/algo-1-1791109825:                       ]8;id=1060208;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060209;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [66]#011train-rmse:0.879611#011validation-rmse:5.51016                                

[10/04/26 10:32:24] INFO     Final Resource Status: Completed                                    ]8;id=1060215;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060216;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31590\31590]8;;\

Now that the training job has finished, we can configure a deployment for data capture, then deploy:

In [6]:
# capture_uri = f's3://{bucket}/data-capture'

# data_capture_config = DataCaptureConfig(
#     enable_capture=True,
#     sampling_percentage=100,
#     destination_s3_uri=capture_uri
# )

from sagemaker.core.shapes import DataCaptureConfig, CaptureOption

capture_uri = f's3://{bucket}/data-capture'

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    initial_sampling_percentage=100,
    destination_s3_uri=capture_uri,
    capture_options=[
        CaptureOption(capture_mode="Input"),
        CaptureOption(capture_mode="Output"),
    ],
)

In [7]:
# xgb_predictor = model.deploy(
#     initial_instance_count=1, instance_type='ml.m4.xlarge',
#     data_capture_config=data_capture_config
# )

from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

model_data = model._latest_training_job.model_artifacts.s3_model_artifacts

# Tear down in dependency order: endpoint -> config -> model
try:
    ep = Endpoint.get(endpoint_name="boston-endpoint")
    ep.delete()
    ep.wait_for_delete()
except Exception:
    pass

try:
    EndpointConfig.get(endpoint_config_name="boston-config").delete()
except Exception:
    pass

try:
    Model.get(model_name="boston-model").delete()
except Exception:
    pass

Model.create(
    model_name="boston-model",
    primary_container=ContainerDefinition(image=algo_image, model_data_url=model_data),
    execution_role_arn=role,
)

EndpointConfig.create(
    endpoint_config_name="boston-config",
    production_variants=[ProductionVariant(
        variant_name="default",
        model_name="boston-model",
        instance_type="ml.m4.xlarge",
        initial_instance_count=1,
    )],
    data_capture_config=data_capture_config,
)

# NOTE: Endpoint.create succeeds (CreateEndpoint is called), but its return
# value blows up: it ends with cls.get(), and the generated Pydantic model
# marks data_capture_config.kms_key_id as required while AWS omits KmsKeyId
# when no KMS key is set. Any resource-layer call that describes this endpoint
# (get / wait_for_status / refresh / invoke) hits the same error.
# Workaround: create + poll + invoke via boto3 for endpoints with data capture.

# xgb_predictor = Endpoint.create(
#     endpoint_name="boston-endpoint",
#     endpoint_config_name="boston-config",
# )

# xgb_predictor.wait_for_status("InService")

import boto3

sm = boto3.client("sagemaker")
sm.create_endpoint(
    EndpointName="boston-endpoint",
    EndpointConfigName="boston-config",
)
sm.get_waiter("endpoint_in_service").wait(EndpointName="boston-endpoint")

[10/04/26 10:32:37] INFO     Creating model resource.                                            ]8;id=1060230;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060231;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#20592\20592]8;;\

                    INFO     Creating endpoint_config resource.                                  ]8;id=1060237;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060238;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11068\11068]8;;\

Here we can provide some sample code to test the deployed model:

In [9]:
# xgb_predictor.serializer = sagemaker.serializers.CSVSerializer()  <- deprecated. There's no predictor to attach a serializer to; you build the body yourself in the next block.

In [10]:
# inputs = test.copy()
# inputs = inputs.drop(columns=inputs.columns[0])

# x_pred = xgb_predictor.predict(inputs.sample(5).values).decode('utf-8')

import io, csv

inputs = test.copy()
inputs = inputs.drop(columns=inputs.columns[0])

buf = io.StringIO()
csv.writer(buf).writerows(inputs.sample(5).values)

# resp = xgb_predictor.invoke(body=buf.getvalue(), content_type="text/csv")  <- In this example, we used boto3 to create the endpoint instead of xgb_predictor = Endpoint.create(). So xgb_predictor does not exist. So we will use boto3 to invoke the endpoint instead.
# x_pred = resp.body.read().decode("utf-8") <- didn't delete it since it would still be useful in case we used xgb_predictor = Endpoint.create() and used the commented out resp variable above.

# Using equivalent of above but with boto3 

rt = boto3.client("sagemaker-runtime")
resp = rt.invoke_endpoint(
    EndpointName="boston-endpoint",
    ContentType="text/csv",
    Body=buf.getvalue(),
)

x_pred = resp["Body"].read().decode("utf-8")
x_pred

'15.302070617675781,33.0619010925293,47.3360595703125,28.042064666748047,22.27298927307129'

We define the Model Monitor and suggest a baseline:

In [20]:
# from sagemaker.model_monitor import DefaultModelMonitor
# from sagemaker.model_monitor.dataset_format import DatasetFormat

# my_monitor = DefaultModelMonitor(     <- deprecated. DefaultModelMonitor doesn't exist and there's no object to hold the config; it moves into the processing job below.
#     role=role,
#     instance_count=1,
#     instance_type='ml.m5.xlarge',
#     volume_size_in_gb=20,
#     max_runtime_in_seconds=3600,
# )

In [11]:
# my_monitor.suggest_baseline(
#     baseline_dataset=f's3://{bucket}/data/train.csv',
#     dataset_format=DatasetFormat.csv(header=False),
# )

from sagemaker.core.resources import ProcessingJob
from sagemaker.core.shapes import (
    ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output,
    ProcessingOutputConfig, ProcessingResources, ProcessingClusterConfig,
    AppSpecification, ProcessingStoppingCondition,
)
import time
job_name = f"boston-baseline-{int(time.time())}"

monitor_image = f"156813124566.dkr.ecr.{region}.amazonaws.com/sagemaker-model-monitor-analyzer"
baseline_results_uri = f"s3://{bucket}/monitoring/baseline"

job = ProcessingJob.create(
    processing_job_name=job_name, # "boston-baseline-01",
    app_specification=AppSpecification(image_uri=monitor_image),
    role_arn=role,
    environment={
        "dataset_format": '{"csv": {"header": false}}',
        "dataset_source": "/opt/ml/processing/input/baseline_dataset_input",
        "output_path": "/opt/ml/processing/output",
        "publish_cloudwatch_metrics": "Disabled",
        "analysis_type": "DATA_QUALITY",
    },
    processing_inputs=[ProcessingInput(
        input_name="baseline_dataset_input",
        s3_input=ProcessingS3Input(
            s3_uri=f"s3://{bucket}/data/train.csv",
            local_path="/opt/ml/processing/input/baseline_dataset_input",
            s3_data_type="S3Prefix",
            s3_input_mode="File",
            s3_data_distribution_type="FullyReplicated",
        ),
    )],
    processing_output_config=ProcessingOutputConfig(outputs=[ProcessingOutput(
        output_name="monitoring_output",
        s3_output=ProcessingS3Output(
            s3_uri=baseline_results_uri,
            local_path="/opt/ml/processing/output",
            s3_upload_mode="EndOfJob",
        ),
    )]),
    processing_resources=ProcessingResources(
        cluster_config=ProcessingClusterConfig(
            instance_count=1,
            instance_type="ml.m5.xlarge",
            volume_size_in_gb=20,
        )
    ),
    stopping_condition=ProcessingStoppingCondition(max_runtime_in_seconds=3600),
)

[10/04/26 10:43:00] INFO     Creating processing_job resource.                                   ]8;id=1060244;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060245;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#28846\28846]8;;\

In [13]:
job.refresh()
print(job.processing_job_status)
print(job.failure_reason)

Completed



Lastly, the Model Monitor must be scheduled, or it won't actually run regular processing jobs on the captured data:

In [14]:
# from sagemaker.model_monitor import CronExpressionGenerator

# my_monitor.create_monitoring_schedule(
#     monitor_schedule_name='my-monitoring-schedule',
#     endpoint_input=xgb_predictor.endpoint_name,
#     statistics=my_monitor.baseline_statistics(),
#     constraints=my_monitor.suggested_constraints(),
#     schedule_cron_expression=CronExpressionGenerator.hourly(),
# )

from sagemaker.core.resources import MonitoringSchedule
from sagemaker.core.shapes import (
    MonitoringScheduleConfig, ScheduleConfig, MonitoringJobDefinition,
    MonitoringInput, EndpointInput, MonitoringOutputConfig, MonitoringOutput,
    MonitoringS3Output, MonitoringResources, MonitoringClusterConfig,
    MonitoringAppSpecification, MonitoringStoppingCondition,
    MonitoringBaselineConfig, MonitoringStatisticsResource,
    MonitoringConstraintsResource,
)

MonitoringSchedule.create(
    monitoring_schedule_name="my-monitoring-schedule",
    monitoring_schedule_config=MonitoringScheduleConfig(
        schedule_config=ScheduleConfig(schedule_expression="cron(0 * ? * * *)"),
        monitoring_job_definition=MonitoringJobDefinition(
            baseline_config=MonitoringBaselineConfig(
                statistics_resource=MonitoringStatisticsResource(
                    s3_uri=f"{baseline_results_uri}/statistics.json"),
                constraints_resource=MonitoringConstraintsResource(
                    s3_uri=f"{baseline_results_uri}/constraints.json"),
            ),
            monitoring_inputs=[MonitoringInput(
                endpoint_input=EndpointInput(
                    endpoint_name="boston-endpoint",
                    local_path="/opt/ml/processing/input/endpoint",
                )
            )],
            monitoring_output_config=MonitoringOutputConfig(
                monitoring_outputs=[MonitoringOutput(
                    s3_output=MonitoringS3Output(
                        s3_uri=f"s3://{bucket}/monitoring/reports",
                        local_path="/opt/ml/processing/output",
                    )
                )]
            ),
            monitoring_resources=MonitoringResources(
                cluster_config=MonitoringClusterConfig(
                    instance_count=1,
                    instance_type="ml.m5.xlarge",
                    volume_size_in_gb=20,
                )
            ),
            monitoring_app_specification=MonitoringAppSpecification(
                image_uri=monitor_image,
            ),
            stopping_condition=MonitoringStoppingCondition(
                max_runtime_in_seconds=3600,
            ),
            role_arn=role,
        ),
    ),
)

[10/04/26 10:52:54] INFO     Creating monitoring_schedule resource.                              ]8;id=1060251;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1060252;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#24562\24562]8;;\

MonitoringSchedule(monitoring_schedule_name='my-monitoring-schedule', monitoring_schedule_arn='arn:aws:sagemaker:us-east-1:864107284170:monitoring-schedule/my-monitoring-schedule', monitoring_schedule_status='Pending', monitoring_type='DataQuality', failure_reason=Unassigned(), creation_time=datetime.datetime(2026, 10, 4, 10, 52, 54, 228000, tzinfo=tzlocal()), last_modified_time=datetime.datetime(2026, 10, 4, 10, 52, 54, 290000, tzinfo=tzlocal()), monitoring_schedule_config=MonitoringScheduleConfig(schedule_config=ScheduleConfig(schedule_expression='cron(0 * ? * * *)', data_analysis_start_time=Unassigned(), data_analysis_end_time=Unassigned()), monitoring_job_definition=MonitoringJobDefinition(monitoring_inputs=[MonitoringInput(endpoint_input=EndpointInput(endpoint_name='boston-endpoint', local_path='/opt/ml/processing/input/endpoint', s3_input_mode='File', s3_data_distribution_type='FullyReplicated', features_attribute=Unassigned(), inference_attribute=Unassigned(), probability_attrib

In [17]:
# Lists each run the schedule has produced. Empty until the first cron boundary
# passes (hourly, on the hour). Each summary carries MonitoringExecutionStatus:
# Completed = ran and found no violations, CompletedWithViolations = drift
# detected against the baseline, Failed = the job itself broke (see
# FailureReason). Reports land under the MonitoringOutputConfig S3 prefix.
sm.list_monitoring_executions(MonitoringScheduleName="my-monitoring-schedule")

{'MonitoringExecutionSummaries': [],
 'ResponseMetadata': {'RequestId': '6a6d2629-2327-4a9a-b1a3-c9f3ea4fb2e9',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '6a6d2629-2327-4a9a-b1a3-c9f3ea4fb2e9',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '35',
   'date': 'Sun, 04 Oct 2026 10:56:18 GMT'},
  'RetryAttempts': 0}}

In [18]:
# Describes the schedule itself, not its runs. MonitoringScheduleStatus should
# be "Scheduled"; "Failed" means the config is bad rather than a run being bad,
# and FailureReason explains why. Also echoes back the full resolved config,
# which is the quickest way to confirm the baseline URIs, endpoint name and
# analyzer image are what you intended.
sm.describe_monitoring_schedule(MonitoringScheduleName="my-monitoring-schedule")

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:864107284170:monitoring-schedule/my-monitoring-schedule',
 'MonitoringScheduleName': 'my-monitoring-schedule',
 'MonitoringScheduleStatus': 'Scheduled',
 'MonitoringType': 'DataQuality',
 'CreationTime': datetime.datetime(2026, 10, 4, 10, 52, 54, 228000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 10, 4, 10, 52, 59, 311000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinition': {'BaselineConfig': {'ConstraintsResource': {'S3Uri': 's3://sagemaker-us-east-1-864107284170/monitoring/baseline/constraints.json'},
    'StatisticsResource': {'S3Uri': 's3://sagemaker-us-east-1-864107284170/monitoring/baseline/statistics.json'}},
   'MonitoringInputs': [{'EndpointInput': {'EndpointName': 'boston-endpoint',
      'LocalPath': '/opt/ml/processing/input/endpoint',
      'S3InputMode': 'File',
      'S3DataDistributionType': 'FullyReplicated

In [20]:
# Cleaning up
#
# AWS deletes are asynchronous: the API accepting a delete does not mean the
# resource is gone. Each stage below waits for describe() to raise before
# moving on, since each resource blocks deletion of the one above it:
#   monitoring schedule -> endpoint -> endpoint config -> model

import boto3, time
sm = boto3.client('sagemaker')


def wait_gone(describe, label, **kw):
    """Poll describe until it raises, i.e. the resource no longer exists."""
    while True:
        try:
            describe(**kw)
            time.sleep(10)
        except Exception:
            print(f"{label} gone")
            return


# 1. Schedule first — an attached schedule blocks endpoint deletion
try:
    sm.delete_monitoring_schedule(MonitoringScheduleName="my-monitoring-schedule")
    print("schedule deleting...")
except Exception as e:
    print("schedule:", e)

wait_gone(sm.describe_monitoring_schedule, "schedule",
          MonitoringScheduleName="my-monitoring-schedule")

# 2. Endpoint — this is the thing that bills continuously
try:
    sm.delete_endpoint(EndpointName="boston-endpoint")
    print("endpoint deleting...")
except Exception as e:
    print("endpoint:", e)

wait_gone(sm.describe_endpoint, "endpoint", EndpointName="boston-endpoint")

# 3. Config, then model — both are instant, no polling needed
for fn, kw in [
    (sm.delete_endpoint_config, {"EndpointConfigName": "boston-config"}),
    (sm.delete_model, {"ModelName": "boston-model"}),
]:
    try:
        fn(**kw)
        print("deleted", kw)
    except Exception as e:
        print(kw, e)

print("done")

schedule: An error occurred (ResourceNotFound) when calling the DeleteMonitoringSchedule operation: Monitoring Schedule arn:aws:sagemaker:us-east-1:864107284170:monitoring-schedule/my-monitoring-schedule not found
schedule gone
endpoint: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Could not find endpoint "boston-endpoint".
endpoint gone
{'EndpointConfigName': 'boston-config'} An error occurred (ValidationException) when calling the DeleteEndpointConfig operation: Could not find endpoint configuration "boston-config".
{'ModelName': 'boston-model'} An error occurred (ValidationException) when calling the DeleteModel operation: Could not find model "boston-model".
done


## Clarify

This Clarify demo builds on the previous demo: we follow the same pattern of define-configure-schedule for our Monitor. Clarify, however, needs more config. We define `SHAPConfig`, `ModelConfig`, `ExplainabilityAnalysisConfig`, and pass them all to the scheduling method.

In [23]:
# model_explainability_monitor = sagemaker.model_monitor.ModelExplainabilityMonitor(
#     role=role,
#     sagemaker_session=session,
#     max_runtime_in_seconds=1800,
# )


# shap_config = sagemaker.clarify.SHAPConfig(
#     baseline=[train.mean().astype(int).to_list()[1:]],
#     num_samples=int(x_train.size),
#     agg_method="mean_abs",
#     save_local_shap_values=False,
# )


# model_config = sagemaker.clarify.ModelConfig(
#     model_name="xgboost-2021-08-25-15-19-33-499",
#     instance_count=1,
#     instance_type='ml.m4.xlarge',
#     content_type="text/csv",
#     accept_type="text/csv",
# )

# analysis_config = sagemaker.model_monitor.ExplainabilityAnalysisConfig(
#         explainability_config=shap_config,
#         model_config=model_config,
#         headers=train.columns.to_list()[1:],
#     )

# explainability_uri = f"s3://{bucket}/model_explainability"
# model_explainability_monitor.create_monitoring_schedule(
#     output_s3_uri=explainability_uri,
#     analysis_config=analysis_config,
#     endpoint_input=xgb_predictor.endpoint_name,
#     schedule_cron_expression=CronExpressionGenerator.hourly(),
# )

In [15]:
# All the code in the code block above is deprecated.
# sagemaker.model_monitor and sagemaker.clarify are removed, and there's no wrapper replacement for either. Those classes existed to generate a JSON config file and launch a processing job with the Clarify image — so in v3 you write the JSON yourself

# 1 - Build and upload the analysis config
import json, boto3

analysis_config = {
    "methods": {
        "shap": {
            "baseline": [train.mean().astype(int).to_list()[1:]],
            "num_samples": int(train.size),
            "agg_method": "mean_abs",
            "save_local_shap_values": False,
        }
    },
    "headers": train.columns.to_list()[1:],
    "predictor": {
        "model_name": "<your-model-name>",
        "instance_type": "ml.m4.xlarge",
        "initial_instance_count": 1,
        "content_type": "text/csv",
        "accept_type": "text/csv",
    },
}

explainability_uri = f"s3://{bucket}/model_explainability"
config_key = "model_explainability/analysis_config.json"
config_uri = f"s3://{bucket}/{config_key}"

boto3.client("s3").put_object(
    Bucket=bucket, Key=config_key, Body=json.dumps(analysis_config).encode()
)

{'ResponseMetadata': {'RequestId': 'AQ0PPJXFA2WBSHR9',
  'HostId': 'AME+QrHsB0HqGqQA/U/lMT8hFSCY1kBdwNB+C6ZDk1LFz/ZtG8Bf0Yj4HMPCrAwfZPrRo+ETMSb8/kMozJYIagKMmRa2PXB4',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'AME+QrHsB0HqGqQA/U/lMT8hFSCY1kBdwNB+C6ZDk1LFz/ZtG8Bf0Yj4HMPCrAwfZPrRo+ETMSb8/kMozJYIagKMmRa2PXB4',
   'x-amz-request-id': 'AQ0PPJXFA2WBSHR9',
   'date': 'Tue, 06 Oct 2026 08:43:18 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"f1c6bc80527135ce934477ef51989a6b"',
   'x-amz-checksum-crc32': 'rbKIkQ==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"f1c6bc80527135ce934477ef51989a6b"',
 'ChecksumCRC32': 'rbKIkQ==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [22]:
# # Debug

# import boto3
# sm = boto3.client('sagemaker')
# try:
#     sm.delete_monitoring_schedule(MonitoringScheduleName="explainability-schedule")
# except Exception as e:
#     print("schedule:", e)

# try:
#     ModelExplainabilityJobDefinition.get(
#         job_definition_name="wine-explainability-def"
#     ).delete()
# except Exception as e:
#     print("delete:", e)

In [23]:
# 2 - Create the schedule
# Same MonitoringSchedule shape as the data-quality one, with two differences: the Clarify image instead of the analyzer, and a second monitoring input pointing at the config file.

from sagemaker.core.resources import ModelExplainabilityJobDefinition, MonitoringSchedule
from sagemaker.core.shapes import (
    ModelExplainabilityAppSpecification, ModelExplainabilityJobInput,
    MonitoringOutputConfig, MonitoringOutput, MonitoringS3Output,
    MonitoringResources, MonitoringClusterConfig, MonitoringStoppingCondition,
    EndpointInput, MonitoringScheduleConfig, ScheduleConfig,
)

clarify_image = f"205585389593.dkr.ecr.{region}.amazonaws.com/sagemaker-clarify-processing:1.0"

ModelExplainabilityJobDefinition.create(
    job_definition_name="wine-explainability-def",
    model_explainability_app_specification=ModelExplainabilityAppSpecification(
        image_uri=clarify_image,
        config_uri=config_uri,
    ),
    model_explainability_job_input=ModelExplainabilityJobInput(
        endpoint_input=EndpointInput(
            endpoint_name="wine-endpoint",
            local_path="/opt/ml/processing/input/endpoint",
        )
    ),
    model_explainability_job_output_config=MonitoringOutputConfig(
        monitoring_outputs=[MonitoringOutput(
            s3_output=MonitoringS3Output(
                s3_uri=explainability_uri,
                local_path="/opt/ml/processing/output",
            )
        )]
    ),
    job_resources=MonitoringResources(
        cluster_config=MonitoringClusterConfig(
            instance_count=1,
            instance_type="ml.m5.xlarge",
            volume_size_in_gb=20,
        )
    ),
    stopping_condition=MonitoringStoppingCondition(max_runtime_in_seconds=1800),
    role_arn=role,
)

MonitoringSchedule.create(
    monitoring_schedule_name="explainability-schedule",
    monitoring_schedule_config=MonitoringScheduleConfig(
        schedule_config=ScheduleConfig(schedule_expression="cron(0 * ? * * *)"),
        monitoring_job_definition_name="wine-explainability-def",
        monitoring_type="ModelExplainability",
    ),
)

[10/06/26 11:23:22] INFO     Creating model_explainability_job_definition resource.              ]8;id=15821307;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15821308;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#22226\22226]8;;\

                    INFO     Creating monitoring_schedule resource.                              ]8;id=15821314;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15821315;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#24562\24562]8;;\

MonitoringSchedule(monitoring_schedule_name='explainability-schedule', monitoring_schedule_arn='arn:aws:sagemaker:us-east-1:864107284170:monitoring-schedule/explainability-schedule', monitoring_schedule_status='Pending', monitoring_type='ModelExplainability', failure_reason=Unassigned(), creation_time=datetime.datetime(2026, 10, 6, 11, 23, 22, 999000, tzinfo=tzlocal()), last_modified_time=datetime.datetime(2026, 10, 6, 11, 23, 23, 71000, tzinfo=tzlocal()), monitoring_schedule_config=MonitoringScheduleConfig(schedule_config=ScheduleConfig(schedule_expression='cron(0 * ? * * *)', data_analysis_start_time=Unassigned(), data_analysis_end_time=Unassigned()), monitoring_job_definition=Unassigned(), monitoring_job_definition_name='wine-explainability-def', monitoring_type='ModelExplainability'), endpoint_name='wine-endpoint', last_monitoring_execution_summary=MonitoringExecutionSummary(monitoring_schedule_name='explainability-schedule', scheduled_time=datetime.datetime(2026, 10, 6, 11, 0, tzi

In [25]:
model_explainability_monitor.delete_monitoring_schedule()


Deleting Monitoring Schedule with name: monitoring-schedule-2021-09-02-22-00-36-508


In [32]:
my_monitor.delete_monitoring_schedule()


Deleting Monitoring Schedule with name: my-monitoring-schedule
